In [1]:
from pathlib import Path
from PIL import Image, ImageDraw
import numpy as np, math, random, json, functools
from tqdm.auto import tqdm
import matplotlib.pyplot as plt

from lightning.pytorch.callbacks import EarlyStopping, ModelCheckpoint
import lightning as L

import torch
import torchmetrics
import torch.nn as nn
import torchvision.transforms.v2 as Tv2
from torchvision import transforms, models
from torchvision.models import resnet18, ResNet18_Weights
from torch.utils.data import Dataset, DataLoader, Sampler
from torchvision import transforms as T


## Download Limited Quick Draw Data

In [2]:
# good default: medium = TF32 on matmul, FP32 on conv
torch.set_float32_matmul_precision("medium")   # or "high" for both ops

# optional: enable TF32 in cuDNN convs as well
torch.backends.cudnn.conv.fp32_precision = "tf32"

/home/sd205521/miniconda3/envs/huggingface/lib/python3.12/site-packages/torch/__init__.py:1612: UserWarning: Please use the new API settings to control TF32 behavior, such as torch.backends.cudnn.conv.fp32_precision = 'tf32' or torch.backends.cuda.matmul.fp32_precision = 'ieee'. Old settings, e.g, torch.backends.cuda.matmul.allow_tf32 = True, torch.backends.cudnn.allow_tf32 = True, allowTF32CuDNN() and allowTF32CuBLAS() will be deprecated after Pytorch 2.9. Please see https://pytorch.org/docs/main/notes/cuda.html#tensorfloat-32-tf32-on-ampere-and-later-devices (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:80.)
  _C._set_float32_matmul_precision(precision)


In [3]:
# # ---------- constants ----------
CLASSES   = ["circle", "triangle", "square", "hexagon", "octagon"]
SIDE_MAP  = {"circle": 0, "triangle": 3, "square": 4, "hexagon": 6, "octagon": 8}
CLASS2IDX = {n: i for i, n in enumerate(CLASSES)}

OUT_RES     = 256
HI_RES      = 112
STROKE_PX   = 6

N_BINS      = 10                          # 0‑10, 10‑20, … 90‑100 %
BIN_EDGES   = np.linspace(0, 100, 11)
N_PER_SHAPE = 50_000                     # images per class
N_PER_BIN   = N_PER_SHAPE // N_BINS

rng = np.random.default_rng(42)

## Synthetic Images

In [4]:
# ───────────── fast synthetic generator ─────────────
import math, functools
import numpy as np
from skimage.draw import polygon_perimeter  # Cython‑accelerated
from skimage.transform import resize as sk_resize
from skimage.morphology import dilation, square   # ← NEW
import math, functools
import numpy as np

OUT_RES   = 256           # keep the public API unchanged
HI_RES    = 64            # draw directly at inference resolution
STROKE    = 10             # px at 64×64 → looks like 6 px at 256×256
rng       = np.random.default_rng(42)

# ---------- vectorised helpers ----------
@functools.lru_cache(maxsize=None)
def _perfect_vertices(n_sides: int) -> np.ndarray:
    """Returns (n,2) array of perfect polygon vertices centred on (0,0)."""
    angles = np.linspace(0, 2*np.pi, n_sides, endpoint=False, dtype=np.float32)
    return np.stack([np.cos(angles), np.sin(angles)], 1)

def _sample_vertices(n_sides: int,
                     reg_pct:  float,
                     n_circle_pts: int = 128) -> np.ndarray:
    """
    Return vertices on the *unit circle* (x,y ∈ [‑1,1]) that describe either
    ‑ a polygon with `n_sides` (3–8), or
    ‑ a distorted circle when `n_sides == 0`.

    For circles (n_sides == 0) the shape is kept smooth:
      • high regularity → almost perfect circle  
      • low regularity  → ellipse / squished blob with 1‑3 gentle bulges
    """
    amp = _curve_amp(reg_pct)                # 0 %→amp_max … 100 %→amp_min

    # ─────── smooth circle / blob branch ────────────────────────────────
    if n_sides == 0:
        base = _perfect_vertices(n_circle_pts)          # surrogate circle

        # ≥50 % regularity  → leave essentially perfect
        if reg_pct >= 50:
            return base

        # distortion strength factor (1 → 3.5 as reg_pct drops 50 → 0)
        k = 1 + (50 - reg_pct) / 20.0

        # 1) global affine: anisotropic stretch + optional shear
        sx    = 1 + rng.normal(0, amp * 1.5 * k)        # x‑stretch
        sy    = 1 + rng.normal(0, amp * 1.5 * k)        # y‑stretch
        shear = rng.normal(0, amp * 0.8 * k)            # x‑shear
        affine = np.array([[sx, shear],
                           [0.0,  sy  ]], dtype=np.float32)
        verts = (affine @ base.T).T                     # (128,2)

        # 2) low‑frequency radial ripple (1‑3 lobes → blobs, not stars)
        n_lobes = rng.integers(1, 4)                    # 1, 2 or 3
        phase   = rng.uniform(0, 2*np.pi)
        ripple  = 1 + amp * 0.6 * k * np.sin(
            np.linspace(phase,
                        phase + n_lobes * 2 * np.pi,
                        n_circle_pts,
                        dtype=np.float32)
        )
        verts *= ripple[:, None]

        # 3) one Chaikin pass for extra smoothness
        return _chaikin(verts, n_iter=1)

    # ─────── polygon branch (unchanged) ─────────────────────────────────
    base   = _perfect_vertices(n_sides)
    radial = rng.normal(1.0, amp, n_sides).astype(np.float32)
    angle  = rng.normal(0.0, amp * 0.6, n_sides).astype(np.float32)
    c, s   = np.cos(angle), np.sin(angle)
    rot    = np.stack([c, s, -s, c], -1).reshape(-1, 2, 2)
    verts  = (rot @ base[:, :, None]).squeeze(-1) * radial[:, None]

    if reg_pct >= 50:
        return verts

    # (same “bulged edges” logic as before)
    bulged, shape_scale = [], min(1.0, n_sides / 6)
    bulge = amp * 0.8 * shape_scale
    for i in range(n_sides):
        v0, v1 = verts[i], verts[(i + 1) % n_sides]
        mid    = (v0 + v1) / 2
        nrm    = np.array([v1[1] - v0[1], v0[0] - v1[0]], np.float32)
        nrm   /= (np.linalg.norm(nrm) + 1e-7)
        bulged.extend([v0, mid + bulge * nrm])

    return _chaikin(np.array(bulged, np.float32), n_iter=1)

def _scale_from_reg(reg_pct: float,
                    r_min: float = 0.20,   # radius = 20 % of canvas height at 0 % regularity
                    r_max: float = 0.40) -> float:   # 40 % at 100 % regularity
    """
    Map target regularity (0‑100 %) to a radius scale factor.
    Lower‑quality tiers → smaller shapes; higher‑quality tiers → full size.
    """
    return r_min + (r_max - r_min) * (reg_pct / 100.0)

def _rasterise(verts: np.ndarray,
               canvas: np.ndarray,
               pct_scale: float = 0.40):         # default keeps old behaviour
    """
    Draw a white outline of the polygon defined by *verts* onto *canvas*.

    verts : (N,2) float32, unit circle coordinates (0 – 1)
    canvas: (H,W) uint8, mutable
    pct_scale : fraction of canvas height to use as radius
    """
    h, w = canvas.shape
    cx = w // 2
    cy = h // 2
    r  = h * pct_scale                       # <-- variable radius
    pts = verts * r + np.array([cx, cy], dtype=np.float32)
    rr, cc = polygon_perimeter(pts[:, 1], pts[:, 0],
                               shape=canvas.shape, clip=True)
    canvas[rr, cc] = 255

def generate_batch(shape_name: str,
                   reg_pct_array: np.ndarray) -> np.ndarray:
    """
    shape_name ∈ {'circle','triangle','square','hexagon','octagon'}
    reg_pct_array shape (B,) – target regularities for each image (0‑100 %).
    Returns uint8 images of shape (B, OUT_RES, OUT_RES).
    """
    side_map = {"circle": 0, "triangle": 3, "square": 4,
                "hexagon": 6, "octagon": 8}
    n_sides  = side_map[shape_name]

    B     = len(reg_pct_array)
    canv  = np.zeros((B, HI_RES, HI_RES), dtype=np.uint8)

    for i, rpct in enumerate(reg_pct_array):
        verts = _sample_vertices(n_sides, rpct)
        size  = _scale_from_reg(rpct)                 # NEW: per‑image radius
        _rasterise(verts, canv[i], pct_scale=size)    # pass scale to rasteriser

    # upscale once, Nearest‑Neighbour → exact CNN input look
    imgs_hi = sk_resize(canv,
                        (B, OUT_RES, OUT_RES),
                        order=0,
                        preserve_range=True).astype(np.uint8)
    return imgs_hi

def _curve_amp(reg_pct: float,
               amp_min: float = 0.0,   # barely visible wiggle at 100 % regularity
               amp_max: float = 0.35,   # huge distortion at 0 % regularity
               gamma:   float = 0.60) -> float:
    """
    Convert regularity (0‑100 %) to a *radial‑jitter amplitude* in the range
    [amp_min, amp_max].  Lower regularity → larger amplitude.
    `gamma < 1` exaggerates the growth so the lowest tiers get *much* wilder.
    """
    frac = 1.0 - reg_pct / 100.0          # 0‑>perfect … 1‑>worst
    return amp_min + (amp_max - amp_min) * (frac ** gamma)

def _chaikin(pts: np.ndarray, n_iter: int = 1) -> np.ndarray:
    """Chaikin corner‑cutting → C¹ smooth poly‑line."""
    for _ in range(n_iter):
        Q = 0.75*pts[:-1] + 0.25*pts[1:]
        R = 0.25*pts[:-1] + 0.75*pts[1:]
        pts = np.vstack([Q.T, R.T]).T.reshape(-1, 2)
    return pts

DATA_PATH   = Path("shapes_data.npz")
HOLD_PATH   = Path("qd_holdout.npz")
CACHE_DIR   = Path("qd_cache"); CACHE_DIR.mkdir(exist_ok=True)

# In‑RAM cache so `np.load` happens **once** per file
_NPZ_CACHE: dict[str, np.lib.npyio.NpzFile] = {}

# Convenience wrapper – loads with mmap the first time only
def load_npz(path: str | Path):
    path = str(path)
    if path not in _NPZ_CACHE:
        _NPZ_CACHE[path] = np.load(path, mmap_mode="r")
    return _NPZ_CACHE[path]




def build_synthetic_dataset(force: bool = False):
    if DATA_PATH.exists() and not force:
        print(f"✅ {DATA_PATH} already present – skipping synthesis")
        return

    imgs, cls, reg = [], [], []
    for cid, name in tqdm(enumerate(CLASSES), total=len(CLASSES)):
        for b in tqdm(range(N_BINS), leave=False):
            tgt = rng.uniform(b*10, (b+1)*10, N_PER_BIN)   # 0‑10 %, … 90‑100 %
            imgs.append(generate_batch(name, tgt))
            cls .append(np.full(N_PER_BIN, cid, np.int8))
            reg .append(tgt)           # use *target* as label – no mis‑binning
    np.savez_compressed(DATA_PATH, imgs=imgs, cls=cls, reg=reg)
    print(f"✅ Wrote {DATA_PATH}  ({len(imgs):,} samples)")

build_synthetic_dataset()

✅ shapes_data.npz already present – skipping synthesis


In [5]:
z = np.load("shapes_data.npz"); regbin = (z["reg"] // 10).astype(int)
for cid,n in enumerate(CLASSES):
    counts = np.bincount(regbin[z["cls"]==cid], minlength=10)
    print(f"{n:8}: {counts}")

circle  : [10000 10000 10000 10000 10000 10000 10000 10000 10000 10000]
triangle: [10000 10000 10000 10000 10000 10000 10000 10000 10000 10000]
square  : [10000 10000 10000 10000 10000 10000 10000 10000 10000 10000]
hexagon : [10000 10000 10000 10000 10000 10000 10000 10000 10000 10000]
octagon : [10000 10000 10000 10000 10000 10000 10000 10000 10000 10000]


In [6]:
# import matplotlib.pyplot as plt, numpy as np
# z       = np.load("shapes_data.npz")
# imgs    = z["imgs"]; cls = z["cls"]; regbin = (z["reg"] // 10).astype(int)
# classes = ["circle","triangle","square","hexagon","octagon"]

# fig, ax = plt.subplots(10, 5, figsize=(5, 10))
# for col, name in enumerate(classes):
#     for row in range(10):
#         pool = np.where((cls == col) & (regbin == row))[0]
#         img  = imgs[np.random.choice(pool)]
#         ax[row, col].imshow(img, cmap="gray")
#         ax[row, col].set_axis_off()
#         if row == 0:
#             ax[row, col].set_title(name, fontsize=8)
# plt.tight_layout()
# plt.show()


In [7]:
CACHE = Path("qd_cache"); CACHE.mkdir(exist_ok=True)

@functools.lru_cache(maxsize=None)                     # ➊  keep in RAM
def _qd_array(shape: str):
    p = CACHE / f"{shape}.npy"
    if not p.exists():
        url = f"https://storage.googleapis.com/quickdraw_dataset/full/numpy_bitmap/{shape}.npy"
        print(f"↓ {url}")
        r = requests.get(url, stream=True); r.raise_for_status()
        with open(p, "wb") as f:
            for chunk in r.iter_content(1 << 16):
                f.write(chunk)
    return np.load(p, mmap_mode="r")

## Google Data

In [8]:
def build_qd_holdout(force: bool = False, per_shape: int = 10_000):
    if HOLD_PATH.exists() and not force:
        print(f"✅ {HOLD_PATH} already present – skipping hold‑out build")
        return
    hold_imgs, hold_cls, hold_reg = [], [], []
    print("⏳ Building QuickDraw hold‑out …")
    for shape in CLASSES:
        src    = _qd_array(shape)
        choice = rng.choice(len(src), HOLD_QD_PER_SHAPE, replace=False)
        for idx in choice:
            arr = src[idx].reshape(28, 28)
            img = Image.fromarray(arr).resize((OUT_RES, OUT_RES), Image.NEAREST)
            hold_imgs.append(np.asarray(img, np.uint8))
            hold_cls.append(CLASS2IDX[shape])
            hold_reg.append(-1.0)
    np.savez_compressed(HOLD_PATH, imgs=hold_imgs, cls=hold_cls, reg=hold_reg)
    print(f"✅ Wrote {HOLD_PATH}  ({len(hold_imgs):,} samples)")

# call the builder (no‑op if file is present)
build_qd_holdout()

✅ qd_holdout.npz already present – skipping hold‑out build


## Dataset

In [9]:
class NPShapes(Dataset):
    def __init__(self, npz="shapes_data.npz", indices=None, tfm=None):
        z = np.load(npz)
        self.imgs, self.cls, self.reg = z["imgs"], z["cls"], z["reg"]
        self.idxs = np.arange(len(self.imgs)) if indices is None else indices
        self.tfm  = tfm
    def __len__(self):
        return len(self.idxs)
    def __getitem__(self, i):
        j = self.idxs[i]
        img = Image.fromarray(self.imgs[j])
        if self.tfm:
            img = self.tfm(img)
        return img, torch.tensor(int(self.cls[j])), torch.tensor(float(self.reg[j]))

## Transforms

In [10]:
# -------------------- TRAIN PIPELINE --------------------
train_tfm = Tv2.Compose([
    # geometry -----------------------------------------------------------
    Tv2.RandomAffine(degrees=15, translate=(0.15, 0.15),
                     scale=(0.8, 1.2), fill=0),

    # QuickDraw‑style artefacts -----------------------------------------
    Tv2.GaussianBlur(kernel_size=3, sigma=(0.2, 1.2)),
    Tv2.RandomErasing(p=0.3, scale=(0.002, 0.01),
                     ratio=(0.2, 5.0), value=0),
    Tv2.ColorJitter(brightness=0.4, contrast=0.4),

    # NEW: replace deprecated ToTensor()
    Tv2.ToImage(),                                     # → Image tensor‑subclass
    Tv2.ToDtype(torch.float32, scale=True),            # → float32 0‑1 range

    # salt‑and‑pepper noise --------------------------------------------
    Tv2.Lambda(
        lambda x: x + (torch.rand_like(x) < 0.02).float()
                    * (2 * torch.rand_like(x) - 1)
    ),

    # normalise to [-1, 1] ---------------------------------------------
    Tv2.Normalize((0.5,), (0.5,)),
])


# -------------------- VALIDATION PIPELINE --------------------
val_tfm = Tv2.Compose([
    Tv2.ToImage(),
    Tv2.ToDtype(torch.float32, scale=True),
    Tv2.Normalize((0.5,), (0.5,)),
])


In [11]:
TRAIN_FRAC = 0.9
NUM_WORKERS = 2
all_npz    = np.load("shapes_data.npz")
N          = all_npz["cls"].shape[0]
perm       = rng.permutation(N)
train_idx  = perm[:int(N * TRAIN_FRAC)]
val_idx    = perm[int(N * TRAIN_FRAC):]

train_ds = NPShapes(indices=train_idx, tfm=train_tfm)
val_ds   = NPShapes(indices=val_idx,   tfm=val_tfm)

train_dl = DataLoader(train_ds, batch_size=1024, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=True)
val_dl   = DataLoader(val_ds, batch_size=1024, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

# ─────────────────────────── 9. Hold‑out loader (QuickDraw) ──────────────────
hold_ds = NPShapes(npz="qd_holdout.npz", tfm=val_tfm)
hold_dl = DataLoader(hold_ds, batch_size=1024, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

In [12]:
class ShapeNet(nn.Module):
    def __init__(self, n_cls=5, dropout_p=0.4):
        super().__init__()
        base = resnet18(weights=None)
        base.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
        self.encoder = nn.Sequential(*list(base.children())[:-2])
        self.gap = nn.AdaptiveAvgPool2d(1)
        self.drop = nn.Dropout(dropout_p)
        dim = base.fc.in_features
        self.cls_head = nn.Linear(dim, n_cls)
        self.reg_head = nn.Sequential(nn.Linear(dim, 128), nn.ReLU(inplace=True), nn.Linear(128, 1))
    def forward(self, x):
        f = self.drop(self.gap(self.encoder(x)).flatten(1))
        return self.cls_head(f), self.reg_head(f).squeeze(1)

In [13]:
class ShapeLit(L.LightningModule):
    def __init__(self, model):
        super().__init__()
        self.model     = model
        self.loss_cls  = nn.CrossEntropyLoss()
        self.loss_reg  = nn.MSELoss()
        self.train_acc = torchmetrics.Accuracy(task="multiclass", num_classes=5)
        self.val_acc   = torchmetrics.Accuracy(task="multiclass", num_classes=5)

    def forward(self, x):
        return self.model(x)

    # ---------- shared step ----------
    def _step(self, batch, stage):
        x, y_cls, y_reg = batch
        logits, reg_pred = self(x)
        cls_loss = self.loss_cls(logits, y_cls)

        mask = y_reg >= 0                       # synthetic only
        reg_loss = self.loss_reg(reg_pred.squeeze()[mask], y_reg[mask]) if mask.any() else 0.0
        loss = cls_loss + reg_loss

        acc  = self.train_acc if stage=="train" else self.val_acc
        acc.update(logits.softmax(-1), y_cls)

        self.log(f"{stage}_loss", loss, prog_bar=True)
        self.log(f"{stage}_cls_loss", cls_loss, prog_bar=False)
        self.log(f"{stage}_reg_loss", reg_loss, prog_bar=False)
        self.log(f"{stage}_acc", acc, prog_bar=True, sync_dist=True)

        return loss

    # ---------- hooks ----------
    def training_step(self, batch, _):
        return self._step(batch, "train")

    def validation_step(self, batch, _):
        self._step(batch, "val")

    def configure_optimizers(self):
        opt = torch.optim.AdamW(self.parameters(), lr=3e-4, weight_decay=1e-4)
        sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode="min",
                                                           patience=2, factor=0.5)
        return {
            "optimizer": opt,
            "lr_scheduler": {
                "scheduler": sched,
                "monitor": "val_loss",
            }
        }

In [14]:
early_stop = EarlyStopping(monitor="val_loss", patience=3, mode="min")
ckpt       = ModelCheckpoint(save_top_k=1, monitor="val_loss", mode="min")

trainer = L.Trainer(
    max_epochs      = 100,
    precision       = "16-mixed",     # FP16 on RTX 5090
    accelerator     = "gpu" if torch.cuda.is_available() else "cpu",
    callbacks       = [early_stop, ckpt],
    log_every_n_steps = 10,
)

model = ShapeNet(n_cls=len(CLASSES), dropout_p=0.4)   # adjust args if you wish
model = model.to("cuda")    # Lightning can also move it for you


Using 16bit Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs


In [ ]:
lit_model = ShapeLit(model)

trainer.fit(lit_model, train_dl, val_dl)

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name      | Type               | Params | Mode 
---------------------------------------------------------
0 | model     | ShapeNet           | 11.2 M | train
1 | loss_cls  | CrossEntropyLoss   | 0      | train
2 | loss_reg  | MSELoss            | 0      | train
3 | train_acc | MulticlassAccuracy | 0      | train
4 | val_acc   | MulticlassAccuracy | 0      | train
---------------------------------------------------------
11.2 M    Trainable params
0         Non-trainable params
11.2 M    Total params
44.954    Total estimated model params size (MB)
78        Modules in train mode
0         Modules in eval mode


Sanity Checking: |                                                                                            …

Training: |                                                                                                   …

## Test

In [ ]:
# ─────────────────────────────────────────────────────────────
# 1. Imports & paths
# ─────────────────────────────────────────────────────────────
import numpy as np
import torch, torch.cuda
import matplotlib.pyplot as plt
import cv2
from PIL import Image
from torchvision import transforms
from pathlib import Path

# ---------- model‑specific ----------
#  ↳  make sure `model` is the same backbone class you used for training
#     e.g.  from shapes_cnn import ShapeNet as model
CKPT_PATH = Path("lightning_logs/version_5/checkpoints") / "epoch=13-step=1974.ckpt"
DEVICE    = "cuda" if torch.cuda.is_available() else "cpu"

# ---------- validation transforms ----------
val_tfm = transforms.Compose([
    transforms.ToTensor(),                 # uint8 → float32 in [0,1]
    transforms.Normalize([0.5], [0.5]),    # same stats used in training
])

# ---------- load Lightning module ----------
lit_model = ShapeLit.load_from_checkpoint(CKPT_PATH, model=model)  # `model` is the backbone class
lit_model.eval().to(DEVICE)

# ─────────────────────────────────────────────────────────────
# 2. Draw a 256 × 256 grayscale image with the mouse
# ─────────────────────────────────────────────────────────────
CANVAS_SIZE = 256
THICKNESS   = 6            # stroke width

canvas  = np.zeros((CANVAS_SIZE, CANVAS_SIZE), dtype=np.uint8)
drawing = False
ix = iy = -1

def _mouse(event, x, y, flags, param):
    global ix, iy, drawing, canvas
    if event == cv2.EVENT_LBUTTONDOWN:
        drawing, ix, iy = True, x, y
    elif event == cv2.EVENT_MOUSEMOVE and drawing:
        cv2.line(canvas, (ix, iy), (x, y), 255, THICKNESS)
        ix, iy = x, y
    elif event == cv2.EVENT_LBUTTONUP:
        drawing = False
        cv2.line(canvas, (ix, iy), (x, y), 255, THICKNESS)

win = "Draw (s=predict  r=reset  q=quit)"
cv2.namedWindow(win)
cv2.setMouseCallback(win, _mouse)

while True:
    cv2.imshow(win, canvas)
    key = cv2.waitKey(1) & 0xFF

    if key == ord("s"):            # ── run inference ──
        if canvas.max() == 0:                      # nothing drawn
            print("Canvas is blank – draw something first.")
            continue
        img_u8 = canvas.copy()
        break
    elif key == ord("r"):
        canvas[:] = 0
    elif key == ord("q"):
        cv2.destroyAllWindows()
        raise SystemExit("Quit without prediction")

cv2.destroyAllWindows()

# ─────────────────────────────────────────────────────────────
# 3. Match training look: down‑scale → up‑scale (pixelate)
# ─────────────────────────────────────────────────────────────
PIX_SIZE  = 64                               # dataset’s native resolution
small     = cv2.resize(img_u8, (PIX_SIZE, PIX_SIZE), interpolation=cv2.INTER_AREA)
img_u8_px = cv2.resize(small,   (CANVAS_SIZE, CANVAS_SIZE), interpolation=cv2.INTER_NEAREST)

# ─────────────────────────────────────────────────────────────
# 4. Pre‑process & move to device
# ─────────────────────────────────────────────────────────────
x = val_tfm(Image.fromarray(img_u8_px)).unsqueeze(0).to(DEVICE)   # shape (1,1,H,W)

# ─────────────────────────────────────────────────────────────
# 5. Inference
# ─────────────────────────────────────────────────────────────
with torch.no_grad(), torch.cuda.amp.autocast(dtype=torch.float16):
    logits, reg_pred = lit_model(x)

probs     = logits.softmax(-1).cpu().squeeze().numpy()            # (C,)
pred_cls  = probs.argmax()
pred_reg  = float(reg_pred.item())
pred_name = CLASSES[pred_cls]

# ─────────────────────────────────────────────────────────────
# 6. Display the result
# ─────────────────────────────────────────────────────────────
plt.figure(figsize=(3, 3))
plt.imshow(img_u8_px, cmap="gray")         # show pixelated input
plt.axis("off")
plt.text(
    2, 16,
    f"{pred_name}  {probs[pred_cls]*100:.1f} %",
    color="white", fontsize=12, weight="bold",
    bbox=dict(facecolor="black", alpha=0.6, pad=3),
)
plt.show()
